# Handout #2: Feed-forward Neural Networks (FFNN)
## Forest Cover Type Classification

This notebook implements the tasks outlined in Handout #2 for the Computational Intelligence course. The objective is to predict forest cover types based on cartographic variables using Feed-forward Neural Networks.

**Student:** [Your Name/Group]
**Dataset:** `ds20.csv`

## 1. Data Loading and Preprocessing

We load the dataset and perform basic preprocessing: scaling numerical features and handling the target variable.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Load dataset
df = pd.read_csv('ds20.csv')
# Fix column names (remove '#' and extra spaces)
df.columns = [c.replace('# ', '').strip() for c in df.columns]
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (7000, 55)


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39,Cover_Type
0,3108,32,3,182,30,1595,219,232,150,3071,...,0,0,0,0,0,0,0,0,0,1
1,2428,63,12,0,0,997,230,216,117,808,...,0,0,0,0,0,0,0,0,0,6
2,2261,321,15,30,10,949,181,224,183,1209,...,0,0,0,0,0,0,0,0,0,6
3,2789,238,22,85,27,993,173,252,210,1650,...,0,0,0,0,0,0,0,0,0,5
4,3087,27,20,722,62,1879,206,191,114,2071,...,0,1,0,0,0,0,0,0,0,1


In [2]:
# Separate features and target
X = df.drop(columns=['Cover_Type']).values
y = df['Cover_Type'].values

# Subtract 1 from labels to make them 0-6 (PyTorch CrossEntropy expects 0-indexed labels)
y = y - 1

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test_scaled)
y_test_t = torch.LongTensor(y_test)

print(f"X_train shape: {X_train_t.shape}")
print(f"Unique labels in y: {np.unique(y)}")

X_train shape: torch.Size([5600, 54])
Unique labels in y: [0 1 2 3 4 5 6]


## 2. Helper Functions for Training

We define a helper function to facilitate the training loop, as PyTorch requires a manual loop.

In [3]:
def train_model(model, X_train, y_train, epochs=50, batch_size=32, lr=0.001, optimizer_type='adam', scheduler=None):
    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    if optimizer_type == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_type == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr)
    
    history = {'loss': [], 'acc': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            
        if scheduler:
            scheduler.step(running_loss/len(loader))
            
        history['loss'].append(running_loss/len(loader))
        history['acc'].append(correct/total)
        
    return history

def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        _, predicted = torch.max(outputs.data, 1)
        acc = accuracy_score(y_test.numpy(), predicted.numpy())
    return acc

## 3. T1: Baseline Model

Starting with a single hidden layer network.

In [4]:
class BaselineModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BaselineModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

model_t1 = BaselineModel(X_train_t.shape[1], 7)
history_t1 = train_model(model_t1, X_train_t, y_train_t, epochs=50, batch_size=32)
acc_t1 = evaluate_model(model_t1, X_test_t, y_test_t)
print(f"T1 Baseline Accuracy: {acc_t1:.4f}")

T1 Baseline Accuracy: 0.7564


## 4. T2-T4: Hyperparameter Tuning

Experimenting with alternative optimizers and activation functions.

In [5]:
# T2: SGD Optimizer
model_sgd = BaselineModel(X_train_t.shape[1], 7)
train_model(model_sgd, X_train_t, y_train_t, epochs=30, optimizer_type='sgd', lr=0.01)
print(f"SGD Test Acc: {evaluate_model(model_sgd, X_test_t, y_test_t):.4f}")

# T3: Activation functions (Tanh)
class TanhModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TanhModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.Tanh(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.network(x)

model_tanh = TanhModel(X_train_t.shape[1], 7)
train_model(model_tanh, X_train_t, y_train_t, epochs=30)
print(f"Tanh Test Acc: {evaluate_model(model_tanh, X_test_t, y_test_t):.4f}")

SGD Test Acc: 0.6836
Tanh Test Acc: 0.7300


## 5. T5-T6: Architecture Expansion

Adding more hidden layers.

In [6]:
# T5: Two hidden layers
class TwoLayerModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TwoLayerModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.network(x)

model_t5 = TwoLayerModel(X_train_t.shape[1], 7)
train_model(model_t5, X_train_t, y_train_t, epochs=50)
print(f"T5 (2 Layers) Test Acc: {evaluate_model(model_t5, X_test_t, y_test_t):.4f}")

# T6: Three hidden layers
class ThreeLayerModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ThreeLayerModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.network(x)

model_t6 = ThreeLayerModel(X_train_t.shape[1], 7)
train_model(model_t6, X_train_t, y_train_t, epochs=50)
print(f"T6 (3 Layers) Test Acc: {evaluate_model(model_t6, X_test_t, y_test_t):.4f}")

T5 (2 Layers) Test Acc: 0.7521
T6 (3 Layers) Test Acc: 0.7514


## 6. T7: Batch Size Sensitivity

Comparing impact of batch size.

In [7]:
import time
batch_sizes = [32, 128, 512]
for b in batch_sizes:
    m = BaselineModel(X_train_t.shape[1], 7)
    start_t = time.time()
    train_model(m, X_train_t, y_train_t, epochs=10, batch_size=b)
    end_t = time.time()
    acc = evaluate_model(m, X_test_t, y_test_t)
    print(f"Batch Size {b}: Acc={acc:.4f}, Time={end_t-start_t:.2f}s")

Batch Size 32: Acc=0.7071, Time=4.00s
Batch Size 128: Acc=0.6650, Time=1.76s
Batch Size 512: Acc=0.6000, Time=1.22s


## 7. T8: Learning Rate Schedule

Implementing a dynamic learning rate.

In [8]:
model_lr = BaselineModel(X_train_t.shape[1], 7)
optimizer = optim.Adam(model_lr.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Custom training call with scheduler
train_model(model_lr, X_train_t, y_train_t, epochs=50, scheduler=scheduler)
print(f"LR Scheduler Test Acc: {evaluate_model(model_lr, X_test_t, y_test_t):.4f}")

LR Scheduler Test Acc: 0.7507


## 8. T9: Final Evaluation (5-Fold CV)

Performing 5-fold cross-validation on our best architecture (3-layer model).

In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_t)):
    X_f_train = X_train_t[train_idx]
    y_f_train = y_train_t[train_idx]
    X_f_val = X_train_t[val_idx]
    y_f_val = y_train_t[val_idx]
    
    model = ThreeLayerModel(X_train_t.shape[1], 7)
    train_model(model, X_f_train, y_f_train, epochs=50, batch_size=64)
    
    acc = evaluate_model(model, X_f_val, y_f_val)
    cv_scores.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

print(f"\nMean Accuracy: {np.mean(cv_scores):.4f}")
print(f"Std Deviation: {np.std(cv_scores):.4f}")

Fold 1 Accuracy: 0.7750
Fold 2 Accuracy: 0.7670
Fold 3 Accuracy: 0.7741
Fold 4 Accuracy: 0.7723
Fold 5 Accuracy: 0.7420

Mean Accuracy: 0.7661
Std Deviation: 0.0124
